# Tagalog Fake News Detection: Weighted Top-2 Ensemble & Individual Model Evaluation
## Repo: https://github.com/joms-hub/tagalog-fake-news-detection


In [1]:
# 1. Load code/data
!git clone https://github.com/joms-hub/tagalog-fake-news-detection.git
import os
os.chdir('/kaggle/working/tagalog-fake-news-detection')

# 2. Install packages (if needed)
!pip install transformers datasets evaluate huggingface_hub accelerate torch

Cloning into 'tagalog-fake-news-detection'...
remote: Enumerating objects: 271, done.
remote: Counting objects: 100% (271/271), done.
remote: Compressing objects: 100% (195/195), done.
remote: Total 271 (delta 153), reused 171 (delta 72), pack-reused 0 (from 0)
Receiving objects: 100% (271/271), 5.69 MiB | 21.05 MiB/s, done.
Resolving deltas: 100% (153/153), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 109.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 78.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━

In [2]:
!pip install --upgrade datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.6/503.6 kB 8.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 46.4 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 19.0.1
    Uninstalling pyarrow-19.0.1:
      Successfully uninstalled pyarrow-19.0.1
  Attempting uninstall: datasets
    Found existing installation: datasets 3.6.0
    Uninstalling datasets-3.6.0:
      Successfully uninstalled datasets-3.6.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.8.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pylibcudf-cu12 25.2.2 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.
cudf-cu12 25.2.2 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you ha

In [3]:
!pip install scikit-learn

In [4]:
import numpy as np
import pandas as pd
import torch
from datasets import load_from_disk
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import json

### ==========================
### 1. Load Tokenized Datasets
### ==========================

In [5]:
base_path = "/kaggle/working/tagalog-fake-news-detection/tokenized"

# Paths for each model's tokenized dataset
model_dataset_paths = {
    "DistilBERT": {
        "train": f"{base_path}/DistilBERT_train",
        "val": f"{base_path}/DistilBERT_val",
        "test": f"{base_path}/DistilBERT_test"
    },
    "TinyBERT": {
        "train": f"{base_path}/TinyBERT_train",
        "val": f"{base_path}/TinyBERT_val",
        "test": f"{base_path}/TinyBERT_test"
    },
    "MobileBERT": {
        "train": f"{base_path}/MobileBERT_train",
        "val": f"{base_path}/MobileBERT_val",
        "test": f"{base_path}/MobileBERT_test"
    },
    "ELECTRA": {
        "train": f"{base_path}/ELECTRA-small_train",
        "val": f"{base_path}/ELECTRA-small_val",
        "test": f"{base_path}/ELECTRA-small_test"
    },
    "MiniLMv2": {
        "train": f"{base_path}/MiniLMv2_train",
        "val": f"{base_path}/MiniLMv2_val",
        "test": f"{base_path}/MiniLMv2_test"
    }
}

# Load all autotokenized datasets for each model
datasets = {}
for model_name, paths in model_dataset_paths.items():
    datasets[model_name] = {
        "train": load_from_disk(paths["train"]),
        "val": load_from_disk(paths["val"]),
        "test": load_from_disk(paths["test"])
    }

# Use the test set labels from one model (all models share the same split)
test_labels = np.array(datasets["DistilBERT"]["test"]["label"])



### ==========================
### 2. Define HuggingFace Model Paths
### ==========================

In [6]:

model_hf_paths = {
    "DistilBERT": "jcunado/distilbert-multilingual-fake-news-filipino",
    "TinyBERT": "jcunado/TinyBERT-tagalog-fake-news",
    "MobileBERT": "jcunado/MobileBERT-tagalog-fake-news",
    "ELECTRA": "jcunado/ELECTRA-small-tagalog-fake-news",
    "MiniLMv2": "jcunado/MiniLMv2-tagalog-fake-news"
}


### ==========================
### 3. Get Softmax Predictions
### ==========================

In [7]:

def get_softmax_preds(model_name_or_path, test_dataset, batch_size=32):
    tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_name_or_path)
    model.eval()
    all_probs = []
    for i in range(0, len(test_dataset), batch_size):
        batch = test_dataset.select(range(i, min(i + batch_size, len(test_dataset))))
        input_ids = torch.tensor(batch["input_ids"])
        attention_mask = torch.tensor(batch["attention_mask"])
        with torch.no_grad():
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            probs = torch.nn.functional.softmax(logits, dim=1)
        all_probs.append(probs.cpu().numpy())
    return np.concatenate(all_probs, axis=0)

### ==========================
### 4. Generate Predictions Per Model
### ==========================

In [8]:
model_probs = {}
for model_name, hf_path in model_hf_paths.items():
    print(f"Evaluating {model_name} ...")
    model_probs[model_name] = get_softmax_preds(hf_path, datasets[model_name]["test"])
print("All model probabilities computed.")


Evaluating DistilBERT ...


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/593 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

All model probabilities computed.


### ==========================
### 5. Top-2 Ensemble (DistilBERT + MobileBERT)
### ==========================

In [9]:
F11 = 0.9647  # MobileBERT validation F1
F12 = 0.9706  # DistilBERT validation F1
w1 = F11 / (F11 + F12)
w2 = F12 / (F11 + F12)
print(f"Ensemble weights: MobileBERT w1={w1:.4f}, DistilBERT w2={w2:.4f}")

p1 = model_probs["MobileBERT"]
p2 = model_probs["DistilBERT"]
ensemble_probs = w1 * p1 + w2 * p2
ensemble_preds = np.argmax(ensemble_probs, axis=1)

Ensemble weights: MobileBERT w1=0.4985, DistilBERT w2=0.5015



### ==========================
### 6. Metrics Calculation
### ==========================

In [11]:
def compute_metrics(y_true, y_pred, probs=None):
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')
    cm = confusion_matrix(y_true, y_pred)
    fpr, tpr, roc_auc = None, None, None
    if probs is not None:
        fpr, tpr, _ = roc_curve(y_true, probs[:, 1])
        roc_auc = auc(fpr, tpr)
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "confusion_matrix": cm.tolist(),
        "roc_auc": roc_auc,
        "fpr": fpr.tolist() if fpr is not None else None,
        "tpr": tpr.tolist() if tpr is not None else None
    }

### ==========================
### 7. Evaluate All Models + Ensemble
### ==========================

In [12]:
results = {}
for model_name in model_probs:
    preds = np.argmax(model_probs[model_name], axis=1)
    results[model_name] = compute_metrics(test_labels, preds, model_probs[model_name])
results["Ensemble_Top2"] = compute_metrics(test_labels, ensemble_preds, ensemble_probs)


### ==========================
### 8. Save Metrics (CSV & JSON)
### ==========================

In [13]:
import os
os.makedirs("results", exist_ok=True)

In [14]:
metrics_table = pd.DataFrame([
    {
        "model": k,
        "accuracy": v["accuracy"],
        "precision": v["precision"],
        "recall": v["recall"],
        "f1": v["f1"],
        "roc_auc": v["roc_auc"]
    } for k, v in results.items()
])
metrics_table.to_csv("results/ensemble_metrics.csv", index=False)
with open("results/ensemble_metrics.json", "w") as f:
    json.dump(results, f, indent=2)

### ==========================
### 9. Plot Confusion Matrices & ROC Curves (PNG)
### ==========================

In [15]:
def plot_confusion_matrix(cm, model_name):
    plt.figure(figsize=(4,4))
    plt.imshow(cm, cmap='Blues')
    plt.title(f"Confusion Matrix: {model_name}")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.xticks([0,1], ["Real", "Fake"])
    plt.yticks([0,1], ["Real", "Fake"])
    plt.colorbar()
    plt.savefig(f"results/{model_name.lower()}_confusion_matrix.png")
    plt.close()

def plot_roc(fpr, tpr, auc_score, model_name):
    plt.figure(figsize=(6,4))
    plt.plot(fpr, tpr, label=f"AUC = {auc_score:.3f}")
    plt.plot([0,1],[0,1],'--',color='gray')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve: {model_name}")
    plt.legend()
    plt.savefig(f"results/{model_name.lower()}_roc_curve.png")
    plt.close()

for name in results:
    plot_confusion_matrix(results[name]["confusion_matrix"], name)
    if results[name]["fpr"] is not None:
        plot_roc(results[name]["fpr"], results[name]["tpr"], results[name]["roc_auc"], name)


### ==========================
### 10. Display Summary Table
### ==========================

In [16]:
print(metrics_table)

           model  accuracy  precision    recall        f1   roc_auc
0     DistilBERT  0.968815   0.969214  0.968815  0.968809  0.990456
1       TinyBERT  0.869023   0.874437  0.869023  0.868528  0.956967
2     MobileBERT  0.954262   0.955801  0.954262  0.954220  0.984561
3        ELECTRA  0.929314   0.929794  0.929314  0.929297  0.980861
4       MiniLMv2  0.908524   0.908640  0.908524  0.908519  0.974637
5  Ensemble_Top2  0.968815   0.968823  0.968815  0.968815  0.990750
